# Synthesis Extraction Pipeline Example

This notebook demonstrates the complete pipeline for extracting structured synthesis procedures from scientific papers.

## Pipeline Overview

The extraction pipeline follows these steps:

```
PDF (bytes)
    ↓
[1. PDF Extraction] → Markdown text + embedded figures (base64)
    ↓
[2. Figure Extraction] → List of FigureInfo objects (segmented, classified)
    ↓
[3. Text Cleaning] → Clean text (remove figures & references)
    ↓
[4. Material Extraction] → List of synthesized materials
    ↓
[5. Synthesis Extraction] → GeneralSynthesisOntology objects (per material)
```

## Prerequisites

Before running this notebook, ensure you have:

1. **Installed the package**: `pip install -e .` from the repository root
2. **Set up environment variables** in a `.env` file:
   - `MISTRAL_API_KEY` (if using Mistral PDF extraction)
   - `OPENAI_API_KEY` (if using OpenAI models)
   - `GOOGLE_API_KEY` (if using Gemini models)
3. **Prepared your PDF files** in a data directory

---
## Setup and Imports

In [ ]:
import base64
import io
import warnings

import dotenv
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from PIL import Image as PILImage

# Load environment variables from .env file
dotenv.load_dotenv()

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

In [ ]:
import sys

print(sys.executable)
print(sys.path)

### Configuration

Set your paper details and data directory here:

In [ ]:
# Configure your paper details
PAPER_NAME = "Do2024Turning"  # PDF filename without .pdf extension
PAPER_ID = "Do2024Turning"  # Unique identifier for the paper
SUPPL_NAME = None  # No supporting information for test

# Directory containing the PDF files
DATA_DIR = "../../data/pdf_papers//superconductor_pdfs-random-sample/selected_papers/0804.0835_0804.0835v2.pdf"


In [ ]:
import os
from pathlib import Path

# Configure your paper details
PAPER_NAME = "Do2024Turning"  # PDF filename without .pdf extension
PAPER_ID = "Do2024Turning"  # Unique identifier for the paper
SUPPL_NAME = "Do2024Turning"  # No supporting information for test

# Get the notebook's directory and build path from there
NOTEBOOK_DIR = Path(os.getcwd())
# If running from repo root, use examples/data/pdf_papers/
# If running from notebook dir, use ../../data/pdf_papers/
if (NOTEBOOK_DIR / "examples" / "data" / "pdf_papers").exists():
    DATA_DIR = str(NOTEBOOK_DIR / "examples" / "data" / "pdf_papers") + "/"
else:
    DATA_DIR = str(NOTEBOOK_DIR / ".." / ".." / "data" / "pdf_papers") + "/"

print(f"Using DATA_DIR: {DATA_DIR}")

---
## Step 1: PDF Extraction

Extract text and figures from PDF files. Two implementations are available:

| Extractor | Pros | Cons |
|-----------|------|------|
| **DoclingPDFExtractor** | Free, open-source, no API key needed | May be slower |
| **MistralPDFExtractor** | Advanced OCR, better quality | Requires API key |

Both extractors implement the `PdfExtractorInterface` and return markdown text with embedded base64 images.

### Option A: Using MistralPDFExtractor (Recommended for quality)

Requires `MISTRAL_API_KEY` environment variable or passed as argument.

In [ ]:
from llm_synthesis.transformers.pdf_extraction import MistralPDFExtractor

# Initialize the PDF extractor
pdf_extractor = MistralPDFExtractor(
    structured=False  # Set True to get JSON structure instead of markdown
)

# Extract main paper
with open(DATA_DIR + PAPER_NAME + ".pdf", "rb") as f:
    pdf_data = f.read()

paper_markdown = pdf_extractor.forward(pdf_data)
print(f"Extracted {len(paper_markdown)} characters from main paper")

In [ ]:
# Extract supporting information (if available)
try:
    with open(DATA_DIR + SUPPL_NAME + ".pdf", "rb") as f:
        suppl_pdf_data = f.read()
    si_markdown = pdf_extractor.forward(suppl_pdf_data)
    print(f"Extracted {len(si_markdown)} characters from SI")
except FileNotFoundError:
    si_markdown = ""
    print("No supporting information file found")

### Option B: Using DoclingPDFExtractor (Free, no API key)

Uncomment and run this cell if you prefer the open-source option:

In [ ]:
# from llm_synthesis.transformers.pdf_extraction import DoclingPDFExtractor
#
# pdf_extractor = DoclingPDFExtractor(
#     pipeline="standard",  # or "fast" for quicker processing
#     table_mode="accurate",
#     use_gpu=True,
#     format="markdown"
# )
#
# with open(DATA_DIR + PAPER_NAME + ".pdf", "rb") as f:
#     paper_markdown = pdf_extractor.forward(f.read())

### Preview the extracted text

In [ ]:
# Display first 2000 characters of the extracted markdown
display(Markdown(paper_markdown[:5000] + "\n\n...[truncated]"))

### Save extracted markdown for later use

In [ ]:
# Save the extracted markdown
with open(DATA_DIR + PAPER_NAME + ".md", "w", errors="replace") as f:
    f.write(paper_markdown)

if si_markdown:
    with open(DATA_DIR + SUPPL_NAME + ".md", "w", errors="replace") as f:
        f.write(si_markdown)

print(f"Saved markdown files to {DATA_DIR}")

---
## Step 2: Figure Extraction

Extract figures from the markdown text. Figures are embedded as base64 data URIs.

Two segmentation backends are available:

| Backend | Model(s) | Output | Pros | Cons |
|---------|----------|--------|------|------|
| **DINO** (default) | Grounding DINO + ResNet-152 | 28 figure classes | More granular classification | Requires 2 models |
| **Florence** | Florence-2 + LoRA | "quantitative plot" / "qualitative plot" | Single model, end-to-end | Binary classification only |

Each extracted figure includes:
- `base64_data`: The image data
- `figure_class`: Classification label
- `quantitative`: Whether it's a quantitative figure (plot, chart, etc.)
- `context_before` / `context_after`: Surrounding text for context
- `figure_reference`: Reference like "Figure 2a"

In [ ]:
from llm_synthesis.transformers.figure_extraction import FigureExtractorMarkdown

# ============================================================================
# Option A: Florence-2 with LoRA - single model for detection + classification
# ============================================================================
# Uses a fine-tuned Florence-2 model from HuggingFace Hub
# Outputs: "quantitative plot" or "qualitative plot"
# Model: https://huggingface.co/amayuelas/plot-visualization-florence-2-lora-32

# Uncomment to use Florence instead:
figure_extractor_florence = FigureExtractorMarkdown(
    segmenter="florence",
    florence_repo_id="amayuelas/plot-visualization-florence-2-lora-32",
)
figures = figure_extractor_florence.forward(paper_markdown)
print(f"Found {len(figures)} figures in the paper (using Florence-2)")

# You can also use a custom LoRA adapter:
# figure_extractor_custom = FigureExtractorMarkdown(
#     segmenter="florence",
#     florence_repo_id="your-username/your-lora-adapter"
# )

In [ ]:
# # ============================================================================
# # Option B: DINO + ResNet (default) - 28 figure classes
# # ============================================================================
# # Uses Grounding DINO for segmentation and ResNet-152 for classification
# # Provides granular figure types: "Bar plots", "Scatter plot", "Microscopy", etc.

# figure_extractor = FigureExtractorMarkdown(segmenter="dino")

# # Extract figures from the markdown
# figures = figure_extractor.forward(paper_markdown)

# print(f"Found {len(figures)} figures in the paper (using DINO + ResNet)")

### Visualize extracted figures

In [ ]:
# Display all extracted figures with their classifications
for idx, figure in enumerate(figures):
    print("=" * 80)
    is_quantitative = (
        "Quantitative" if figure.quantitative else "Not Quantitative"
    )
    print(f"Figure {idx + 1}: {is_quantitative} - Class: {figure.figure_class}")
    print(f"Reference: {figure.figure_reference}")

    # Decode and display the image
    image_data = base64.b64decode(figure.base64_data)
    image = PILImage.open(io.BytesIO(image_data))

    plt.figure(figsize=(8, 6))
    plt.imshow(np.array(image))
    plt.axis("off")
    plt.title(f"{is_quantitative}: {figure.figure_class}")
    # plt.show()

### Inspect figure metadata

In [ ]:
# Print metadata for the first figure
if figures:
    fig = figures[0]
    print("Figure Reference:", fig.figure_reference)
    print("Figure Class:", fig.figure_class)
    print("Quantitative:", fig.quantitative)
    print("Position in text:", fig.position)
    print("\nAlt Text:", fig.alt_text[:200] if fig.alt_text else "None")
    print(
        "\nContext Before:",
        fig.context_before[:200] if fig.context_before else "None",
    )
    print(
        "\nContext After:",
        fig.context_after[:200] if fig.context_after else "None",
    )

---
## Step 3: Text Cleaning

Clean the extracted text by removing figures and references for better material/synthesis extraction.

Available utilities:
- `clean_text()`: Removes both figures and references
- `remove_figs()`: Removes only figure markdown
- `remove_references()`: Removes only reference sections

In [ ]:
from llm_synthesis.utils.markdown_utils import clean_text

# Clean the paper text
clean_paper_text = clean_text(paper_markdown)
clean_si_text = clean_text(si_markdown) if si_markdown else ""

print(f"Original length: {len(paper_markdown)} characters")
print(f"Cleaned length: {len(clean_paper_text)} characters")
print(f"Removed: {len(paper_markdown) - len(clean_paper_text)} characters")

### Create Paper object

The `Paper` model holds the paper metadata and text content:

In [ ]:
from llm_synthesis.models.paper import Paper

paper = Paper(
    id=PAPER_ID,
    name=PAPER_NAME,
    publication_text=clean_paper_text,
    si_text=clean_si_text,
)

print(f"Created Paper object: {paper.name} (ID: {paper.id})")

---
## Step 4: Material Extraction

Extract the list of synthesized materials from the paper text using LLM-based extraction.

The extractor uses DSPy with configurable LLM backends:
- `gemini-2.0-flash`, `gemini-2.5-flash`, `gemini-2.5-pro` (Google)
- `gpt-4o`, `gpt-4o-mini` (OpenAI)
- `mistral-small`, `mistral-medium`, `mistral-large` (Mistral)

In [ ]:
from llm_synthesis.transformers.material_extraction import (
    DspyTextExtractor,
    make_dspy_text_extractor_signature,
)
from llm_synthesis.utils.dspy_utils import get_llm_from_name

# Create a signature for material extraction
material_signature = make_dspy_text_extractor_signature(
    signature_name="TextToMaterials",
    instructions=(
        "Extract ONLY the final synthesized materials from the publication text. "
        "Focus on materials that were actually synthesized in the paper, not just mentioned. "
        "Return materials as chemical formulas when possible."
    ),
    input_description="The publication text to extract synthesized materials from.",
    output_name="materials",
    output_description=(
        "The final synthesized materials as a comma-separated list using chemical formulas "
        "(e.g., 'Ru/CeO2, Ni-Mo/Al2O3'). Never use common names alone."
    ),
)

# Initialize the LLM
material_lm = get_llm_from_name(
    llm_name="gemini-3.0-pro",  # Fast and cost-effective
    model_kwargs={"temperature": 0.0},
)

# Create the material extractor
material_extractor = DspyTextExtractor(
    signature=material_signature,
    lm=material_lm,
)

In [ ]:
# Extract materials from the paper
materials_text = material_extractor.forward(input=clean_paper_text)

# Parse the comma-separated list into individual materials
materials = [
    material.strip()
    for material in materials_text.replace("\n", ",").split(",")
    if material.strip()
]

print(f"Found {len(materials)} materials:")
for i, material in enumerate(materials, 1):
    print(f"  {i}. {material}")

---
## Step 5: Synthesis Extraction

Extract structured synthesis procedures for each material.

The output is a `GeneralSynthesisOntology` object containing:
- **target_compound**: The material being synthesized
- **target_compound_type**: Category (e.g., "nanomaterials", "ceramics")
- **synthesis_method**: Method used (e.g., "hydrothermal", "sol-gel")
- **starting_materials**: List of precursors with amounts
- **steps**: Ordered list of synthesis steps with conditions
- **equipment**: List of equipment used

**Note:** We use `gpt-4o-mini` here as it has better support for structured JSON outputs. 
Gemini models may fail with Pydantic validation errors for the enum fields.

In [ ]:
from llm_synthesis.transformers.synthesis_extraction import (
    DspySynthesisExtractor,
    make_dspy_synthesis_extractor_signature,
)

# System prompt to guide the model on valid enum values
SYNTHESIS_SYSTEM_PROMPT = """You are a helpful assistant that extracts structured synthesis procedures from scientific papers.

IMPORTANT: For the synthesis_method field, you MUST choose from these exact values:
'PVD', 'CVD', 'arc discharge', 'ball milling', 'spray pyrolysis', 'electrospinning', 
'sol-gel', 'hydrothermal', 'solvothermal', 'precipitation', 'coprecipitation', 'combustion', 
'microwave-assisted', 'sonochemical', 'template-directed', 'solid-state', 'flux growth', 
'float zone & Bridgman', 'arc melting & induction melting', 'spark plasma sintering', 
'electrochemical deposition', 'chemical bath deposition', 'liquid-phase epitaxy', 'self-assembly', 
'atomic layer deposition', 'molecular beam epitaxy', 'pulsed laser deposition', 'ion implantation', 
'lithographic patterning', 'wet impregnation', 'incipient wetness impregnation', 'mechanical mixing', 
'solution-based', 'mechanochemical', 'other'

For the target_compound_type field, you MUST choose from these exact values:
'metals & alloys', 'ceramics & glasses', 'polymers & soft matter', 'composites', 
'semiconductors & electronic', 'nanomaterials', 'two-dimensional materials', 
'framework & porous materials', 'biomaterials & biological', 'liquid materials', 
'hybrid & organic-inorganic', 'functional materials & catalysts', 'energy & sustainability', 
'smart & responsive materials', 'emerging & quantum materials', 'other'

If the exact method is not in the list, use the closest match or 'other'."""

# Create a signature for synthesis extraction
synthesis_signature = make_dspy_synthesis_extractor_signature(
    signature_name="ExtractStructuredSynthesis",
    instructions=(
        "Extract the complete structured synthesis procedure for the specified material. "
        "Include all steps, conditions (temperature, time, atmosphere), equipment, and precursors. "
        "Be thorough and preserve all quantitative details."
    ),
    paper_text_description="The complete paper text containing synthesis procedures.",
    material_name_description="The specific material to extract the synthesis procedure for.",
    output_name="structured_synthesis",
    output_description="The complete structured synthesis procedure for the material.",
)

# Initialize the LLM - use gpt-4o-mini for better structured output support
synthesis_lm = get_llm_from_name(
    llm_name="gemini-3.0-flash",  # Better structured output support than Gemini"gemini-2.5-flash"
    model_kwargs={
        "temperature": 0.0,
        "max_tokens": 60000,
    },
    system_prompt=SYNTHESIS_SYSTEM_PROMPT,
)

# Create the synthesis extractor
synthesis_extractor = DspySynthesisExtractor(
    signature=synthesis_signature,
    lm=synthesis_lm,
)

In [ ]:
from llm_synthesis.models.paper import SynthesisEntry

# Extract synthesis for each material
all_syntheses = []

for material in materials:
    print(f"\nExtracting synthesis for: {material}")
    print("-" * 50)

    try:
        # The extractor takes a tuple of (paper_text, material_name)
        synthesis = synthesis_extractor.forward(
            input=(clean_paper_text, material)
        )

        # Store the result
        all_syntheses.append(
            SynthesisEntry(
                material=material,
                synthesis=synthesis,
                evaluation=None,  # Optional: add judge evaluation
            )
        )

        print(f"  Method: {synthesis.synthesis_method}")
        print(f"  Type: {synthesis.target_compound_type}")
        print(f"  Steps: {len(synthesis.steps)}")
        print(f"  Starting materials: {len(synthesis.starting_materials)}")

    except Exception as e:
        print(f"  Error: {e}")

print(f"\nSuccessfully extracted {len(all_syntheses)} synthesis procedures")

In [ ]:
import dspy

dspy.inspect_history(5)

### Inspect a synthesis result

In [ ]:
# Display the first synthesis in detail
if all_syntheses:
    synthesis = all_syntheses[1].synthesis

    print(f"Target Compound: {synthesis.target_compound}")
    print(f"Compound Type: {synthesis.target_compound_type}")
    print(f"Synthesis Method: {synthesis.synthesis_method}")

    print("\nStarting Materials:")
    for mat in synthesis.starting_materials:
        print(f"  - {mat.name}: {mat.amount} {mat.unit}")

    print("\nSynthesis Steps:")
    for step in synthesis.steps:
        print(f"  {step.step_number}. {step.action}")
        if step.conditions:
            if step.conditions.temperature:
                print(
                    f"      Temperature: {step.conditions.temperature} {step.conditions.temp_unit}"
                )
            if step.conditions.duration:
                print(
                    f"      Duration: {step.conditions.duration} {step.conditions.time_unit}"
                )

    print("\nEquipment:")
    for eq in synthesis.equipment:
        print(f"  - {eq.name}")

### Export synthesis as JSON

In [ ]:
import json

# Export all syntheses to JSON
if all_syntheses:
    synthesis_dict = all_syntheses[0].synthesis.model_dump()
    print(json.dumps(synthesis_dict, indent=2))

---
## Step 6: Create Final Result Object

Combine all extracted data into a `PaperWithSynthesisOntologies` object:

In [ ]:
from llm_synthesis.models.paper import PaperWithSynthesisOntologies

# Create the final result object
result = PaperWithSynthesisOntologies(
    name=paper.name,
    id=paper.id,
    publication_text=paper.publication_text,
    si_text=paper.si_text,
    all_syntheses=all_syntheses,
    cost_data=None,  # Cost tracking is available in production scripts
)

print(f"Created result for paper: {result.name}")
print(f"Total materials: {len(result.all_syntheses)}")
print(f"Total figures: {len(figures)}")

### Save results

In [ ]:
import json
import os

# Save the complete result as JSON
output_dir = DATA_DIR + "results/"
os.makedirs(output_dir, exist_ok=True)

output_path = output_dir + PAPER_ID + ".json"
with open(output_path, "w") as f:
    json.dump(result.model_dump(), f, indent=2)

print(f"Saved results to: {output_path}")

---
## Figure Data Extraction with Claude

Extract numerical coordinate data from quantitative figures (plots, charts) using Claude's vision API.

This is useful for:
- Extracting data points from line plots, scatter plots, bar charts
- Digitizing figure data for further analysis
- Converting visual data back to tabular format

**Requirements:** `ANTHROPIC_API_KEY` environment variable must be set.

In [ ]:
from llm_synthesis.models.figure import FigureInfoWithPaper
from llm_synthesis.transformers.plot_extraction.claude_extraction.plot_data_extraction import (
    ClaudeLinePlotDataExtractor,
)

# Initialize the Claude coordinate extractor
# Available models: "claude-sonnet-4-20250514", "claude-3-5-haiku-latest", etc.
coordinate_extractor = ClaudeLinePlotDataExtractor(
    model_name="claude-sonnet-4-20250514",
    max_tokens=2048,
    temperature=0.0,
)

print("Claude coordinate extractor initialized")

In [ ]:
# Select a quantitative figure for data extraction
# Filter to only quantitative figures (plots, charts, etc.)
quantitative_figures = [f for f in figures if f.quantitative]

print(f"Found {len(quantitative_figures)} quantitative figures:")
for i, fig in enumerate(quantitative_figures):
    print(f"  [{i}] {fig.figure_reference} - {fig.figure_class}")

# Select which figure to extract data from (change index as needed)
FIGURE_INDEX = 0  # Change this to select a different figure

In [ ]:
# Display the selected figure
if quantitative_figures:
    selected_figure = quantitative_figures[FIGURE_INDEX]

    print(
        f"Selected: {selected_figure.figure_reference} - {selected_figure.figure_class}"
    )
    print(
        f"Context: {selected_figure.context_before[:200] if selected_figure.context_before else 'None'}..."
    )

    # Display the figure
    image_data = base64.b64decode(selected_figure.base64_data)
    image = PILImage.open(io.BytesIO(image_data))

    plt.figure(figsize=(10, 8))
    plt.imshow(np.array(image))
    plt.axis("off")
    plt.title(
        f"{selected_figure.figure_reference}: {selected_figure.figure_class}"
    )
    plt.show()
else:
    print("No quantitative figures found in this paper")

### Extract coordinate data from the figure

Now we'll use Claude's vision API to extract the numerical data points from the plot:

In [ ]:
# Extract coordinate data from the selected figure
if quantitative_figures:
    selected_figure = quantitative_figures[FIGURE_INDEX]

    # Create FigureInfoWithPaper object (required by the extractor)
    figure_with_context = FigureInfoWithPaper(
        base64_data=selected_figure.base64_data,
        alt_text=selected_figure.alt_text or "",
        position=selected_figure.position,
        context_before=selected_figure.context_before or "",
        context_after=selected_figure.context_after or "",
        figure_reference=selected_figure.figure_reference or "",
        figure_class=selected_figure.figure_class or "",
        quantitative=selected_figure.quantitative,
        paper_text=clean_paper_text,
        si_text=clean_si_text,
    )

    print("Extracting coordinate data with Claude...")
    print("This may take a few seconds...\n")

    # Extract the data
    extracted_plot_data = coordinate_extractor.forward(figure_with_context)

    # Print extraction cost
    cost = coordinate_extractor.get_cost()
    print(f"Extraction cost: ${cost:.4f}")
else:
    print("No quantitative figures to extract data from")

### View extracted plot metadata and data

In [ ]:
# Display extracted plot metadata
if quantitative_figures and extracted_plot_data:
    print("=== Extracted Plot Metadata ===")
    print(f"Title: {extracted_plot_data.title}")
    print(
        f"X-axis: {extracted_plot_data.x_axis_label} ({extracted_plot_data.x_axis_unit})"
    )
    print(
        f"Y-axis (left): {extracted_plot_data.y_left_axis_label} ({extracted_plot_data.y_left_axis_unit})"
    )

    print("\n=== Extracted Data Series ===")
    for (
        series_name,
        coordinates,
    ) in extracted_plot_data.name_to_coordinates.items():
        print(f"\n{series_name}: {len(coordinates)} data points")
        # Show first 5 points as preview
        for i, (x, y) in enumerate(coordinates[:5]):
            print(f"  ({x}, {y})")
        if len(coordinates) > 5:
            print(f"  ... and {len(coordinates) - 5} more points")

### Visualize extracted data

Plot the extracted data points to verify accuracy:

In [ ]:
# Visualize the extracted data
if (
    quantitative_figures
    and extracted_plot_data
    and extracted_plot_data.name_to_coordinates
):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: Original figure
    image_data = base64.b64decode(selected_figure.base64_data)
    image = PILImage.open(io.BytesIO(image_data))
    axes[0].imshow(np.array(image))
    axes[0].set_title("Original Figure")
    axes[0].axis("off")

    # Right: Extracted data points
    colors = plt.cm.tab10.colors
    for i, (series_name, coordinates) in enumerate(
        extracted_plot_data.name_to_coordinates.items()
    ):
        if coordinates:
            x_vals = [coord[0] for coord in coordinates]
            y_vals = [coord[1] for coord in coordinates]
            axes[1].plot(
                x_vals,
                y_vals,
                "o-",
                color=colors[i % len(colors)],
                label=series_name,
                markersize=4,
            )

    axes[1].set_xlabel(
        f"{extracted_plot_data.x_axis_label or 'X'} ({extracted_plot_data.x_axis_unit or ''})"
    )
    axes[1].set_ylabel(
        f"{extracted_plot_data.y_left_axis_label or 'Y'} ({extracted_plot_data.y_left_axis_unit or ''})"
    )
    axes[1].set_title("Extracted Data Points")
    axes[1].legend(loc="best", fontsize=8)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No data to visualize")

### Export extracted data to CSV

In [ ]:
import csv

# Export extracted data to CSV
if (
    quantitative_figures
    and extracted_plot_data
    and extracted_plot_data.name_to_coordinates
):
    csv_path = output_dir + f"{PAPER_ID}_figure_{FIGURE_INDEX}_data.csv"

    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)

        # Write header
        writer.writerow(
            [
                "series_name",
                f"x ({extracted_plot_data.x_axis_unit or 'units'})",
                f"y ({extracted_plot_data.y_left_axis_unit or 'units'})",
            ]
        )

        # Write data
        for (
            series_name,
            coordinates,
        ) in extracted_plot_data.name_to_coordinates.items():
            for x, y in coordinates:
                writer.writerow([series_name, x, y])

    print(f"Exported data to: {csv_path}")

    # Also save as JSON for complete metadata
    json_path = output_dir + f"{PAPER_ID}_figure_{FIGURE_INDEX}_data.json"
    with open(json_path, "w") as f:
        json.dump(extracted_plot_data.model_dump(), f, indent=2)
    print(f"Exported metadata to: {json_path}")
else:
    print("No data to export")

---
## Summary

This notebook demonstrated the complete synthesis extraction pipeline:

1. **PDF Extraction**: Convert PDF to markdown with embedded figures
2. **Figure Extraction**: Extract and classify figures
3. **Text Cleaning**: Remove figures and references for cleaner processing
4. **Material Extraction**: Identify synthesized materials using LLM
5. **Synthesis Extraction**: Extract structured synthesis procedures per material

For production use, see the deployment script at:
`examples/scripts/deployment/extract_synthesis_procedure_from_text.py`

which includes:
- Parallel processing of multiple papers
- Cost tracking for LLM API calls
- Hydra configuration management
- Synthesis evaluation/judging